In [7]:
import pandas as pd
import numpy as np
from groq import Groq
import mysql.connector
import json
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# DB Connection
conn = mysql.connector.connect(
    host="host_name",
    port="port_number",
    user="username",
    password="password",
    database="database_name"
)

# Groq Client
client = Groq(
    api_key="groq_api_key"
)

# NL2MQL2SQL

In [9]:
# chat to ask
question = input("Question: ")

# Generate MQL
mql_prompt = f"""
Convert the question into MQL JSON.

Available Metrics:
- revenue = revenue_amount
- quantity = quantity

Available Dimensions:
- year
- month
- state
- category
- sub_category
- product_id
- product_description

Return JSON only.

Example:

{{
{{
"analysis_type": "ranking",
"metric":"revenue",
"aggregation": "average",
"group_by":["state"],
"filters":{{"year":2025}}
}}
}}

Question:
{question}
"""

mql_response = client.chat.completions.create(
model="openai/gpt-oss-120b",
messages=[
    {"role": "user", "content": mql_prompt}
    ]
)

mql = (
mql_response.choices[0]
.message.content
.replace("```json", "")
.replace("```", "")
.strip()
)

print("\nMQL Generated:")
print(mql)

# Generate SQL
sql_prompt = f"""
You are an expert SQL generator.

Table: dummy_revenue_sales

Columns:
- year
- month
- id
- product_id
- product_description
- state
- category
- sub_category
- quantity
- price_per_unit
- revenue_amount

MQL:
{mql}

Return SQL only.

Requirements:
- syntax only
- No explanation
- No markdown
- No comments
- SELECT query only
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": sql_prompt}
    ]
)

sql = response.choices[0].message.content

sql = sql.replace("```sql", "").replace("```", "").strip()

print("\nSQL Generated:")
print(sql)

# Safety
if not sql.upper().startswith("SELECT"):
    raise Exception("Only SELECT allowed")

# Execute
cursor = conn.cursor(dictionary=True)

cursor.execute(sql)

rows = cursor.fetchall()

df = pd.DataFrame(rows)

# Convert to business answer
answer_prompt = f"""
User Question:
{question}

Query Result:
{df.to_json(orient='records')}

You are an expert business analystics, provide a concise business answer with natural language without slogan, short and precise.
"""

answer = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": answer_prompt}
    ]
)

print("\nAnswer:")
print(answer.choices[0].message.content)

Question:  which product has highest revenue on month August?



MQL Generated:
{
  "analysis_type": "ranking",
  "metric": "revenue",
  "aggregation": "sum",
  "group_by": ["product_id"],
  "filters": { "month": 8 }
}

SQL Generated:
SELECT product_id, revenue, RANK() OVER (ORDER BY revenue DESC) AS revenue_rank FROM (SELECT product_id, SUM(revenue_amount) AS revenue FROM dummy_revenue_sales WHERE month = 8 GROUP BY product_id) t

Answer:
In August, product **P061** generated the highest revenue, amounting to **$3,600**.


In [ ]:
# CLEANUP
conn.close()